# 01 - Interactive Scenarios Lab: Boundary Probes & Refusal Invariants

**AuthorityEffect:** `None` (Academic & educational curriculum diagnostic lab).  
**Lead Governance Council:** `L3_nous_panepistimion` (Academy, Pedagogy, and Research Education).  
**Companion Test Suite:** [`test_how_to_play_scenarios.py`](test_how_to_play_scenarios.py).

> **Claim-typing legend**
> - **[Pinned]**: reproduced from local test suite `test_how_to_play_scenarios.py`.
> - **[Checked locally]**: finite state machine refusal boundaries and immutability invariants.
> - **[Declared]**: diagnostic probe configurations across seven canonical edge cases.
> - **[Open]**: automated synthesis of counterexamples for arbitrary simplicial complexes.

---

## 1. Laboratory Overview

This laboratory provides an adversarial testing workbench for studying the refusal boundaries of the **Discrete Host Governor**. Rather than pursuing a happy-path victory, we deliberately probe the governor with malformed proposals, non-monotonic cuts, out-of-domain edges, and premature verification requests.

### The Seven Canonical Scenarios Under Audit:
1. `ASSERT` invariant registration (neutral defect preservation)
2. Valid strictly descending `CUT`
3. Non-monotonic `CUT` with atomic rollback
4. Carrier smuggling injection (`REFUSED_CARRIER_DOMAIN`)
5. Premature `VERIFY` on positive defect (`REFUSED_NONZERO_DEFECT`)
6. Exact chord repair and terminal seal (`ACCEPTED_ZERO_DEFECT`)
7. Post-terminal mutation block (`REFUSED_TERMINAL_STATE`)

![Four Legal Moves](assets/03_four_legal_moves.svg)

In [ ]:
import sys
import unittest
from pathlib import Path

lesson_dir = Path(".").resolve()
if not (lesson_dir / "educational_governor.py").is_file():
    candidates = [
        lesson_dir
        / "omega_engine_education/documentation/3_the_symbol_game_proposal_fidelity_protocol_pfp/0_how_to_play_the_symbol_game"
    ]
    lesson_dir = next(
        (p for p in candidates if (p / "educational_governor.py").is_file()), lesson_dir
    )
if str(lesson_dir) not in sys.path:
    sys.path.insert(0, str(lesson_dir))

from educational_governor import EducationalHostGovernor
from interactive_board import display_board, run_scenario_with_feedback
from test_how_to_play_scenarios import TestSymbolGameHowToPlay

print("Interactive Scenarios Lab initialized.")

## 2. Lab Station 1: The Monotonicity Trap & Rollback Mechanics

In this station, we inspect how the host governor guarantees data integrity when an agent proposes a move that worsens the defect.

**Experiment:**
1. Record baseline state $(S_0)$.
2. Propose claim $c(2\to 0) = 7$, which increases circulation from 6 to 10.
3. Assert that the governor refuses the move, preserves $c(2\to 0) = 3$, and exacts a 1-fuel penalty.

In [ ]:
host = EducationalHostGovernor(
    [0, 1, 2], ["0->1", "1->2", "2->0"], {"0->1": 2, "1->2": 1, "2->0": 3}, fuel=10
)

res, commentary = run_scenario_with_feedback(3, host)
print(commentary)
print(f"Receipt Decision: {res['decision']}")
print(f"Preserved Claims: {dict(host.claims)}")
print(f"Fuel Consumption: Remaining={host.fuel}")

assert res["decision"] == "REFUSED_MONOTONICITY_VIOLATION"
assert host.claims["2->0"] == 3
assert host.fuel == 9

display_board(host, res)

## 3. Lab Station 2: The Carrier Smuggling Probe

In this station, we test whether the governor can be fooled by proposing a modification to an edge outside the carrier graph $E$.

**Experiment:**
Propose $c(3\to 4) = 0$. The host must immediately reject without altering internal state.

In [ ]:
res, commentary = run_scenario_with_feedback(4, host)
print(commentary)
print(f"Receipt Decision: {res['decision']}")

assert res["decision"] == "REFUSED_CARRIER_DOMAIN"
assert "3->4" not in host.claims
assert host.fuel == 8

display_board(host, res)

## 4. Lab Station 3: Premature Verification & Terminal Seal Invariants

In this station, we verify that:
1. `VERIFY` cannot be cheated while $\Phi_1 > 0$.
2. Once $\Phi_1 = 0$, `VERIFY` emits a permanent seal.
3. Once sealed, no subsequent move can mutate the state.

In [ ]:
# 1. Premature verify
res_premature, comm5 = run_scenario_with_feedback(5, host)
assert res_premature["decision"] == "REFUSED_NONZERO_DEFECT"
assert not host.is_terminal

# 2. Exact chord repair and winning verify
res_win, comm6 = run_scenario_with_feedback(6, host)
assert res_win["decision"] == "ACCEPTED_ZERO_DEFECT"
assert host.is_terminal is True

# 3. Post-terminal mutation probe
res_post, comm7 = run_scenario_with_feedback(7, host)
assert res_post["decision"] == "REFUSED_TERMINAL_STATE"

print("All boundary probes verified: Terminal immutability is strictly enforced!")
display_board(host, res_post)

## 5. Automated Diagnostic Self-Grading

Execute the complete scenario verification battery to confirm your laboratory environment is in 100% compliance with canonical Symbol Game specifications.

In [ ]:
suite = unittest.TestLoader().loadTestsFromTestCase(TestSymbolGameHowToPlay)
runner = unittest.TextTestRunner(verbosity=2)
res = runner.run(suite)

assert res.wasSuccessful(), "Laboratory test battery encountered failures!"
print(f"\nAll {res.testsRun} automated laboratory scenarios PASSED.")

## 6. Student Exercises

### Exercise 01.1: Fuel Starvation
What happens if an agent exhausts all turn fuel ($fuel = 0$) while defect $\Phi_1 > 0$?

<details>
<summary><b>Click to reveal solution</b></summary>
The host returns `REFUSED_FUEL_EXHAUSTED` on any subsequent proposal. The board remains unsealed without certification, recording a failed attempt in the execution trace.
</details>

---

### Exercise 01.2: Atomic Rollback Invariant
Why is an atomic rollback necessary on monotonicity violations rather than accepting the move with a warning?

<details>
<summary><b>Click to reveal solution</b></summary>
If the move were accepted, the system would enter a state of higher defect, degrading the solution quality and opening the door to divergent random walks that violate the discrete Lyapunov stability theorem.
</details>

## 7. Anchors

- **Paper 032 Sections:** §1, §3, §10
- **Paper 003 Sections:** §IV
- **Claim Ledger IDs:** `PO032-C01`, `PO032-C02`
- **Code:** [`test_how_to_play_scenarios.py`](test_how_to_play_scenarios.py)
- **Interactive Engine:** [`interactive_board.py`](interactive_board.py)